# Batch Write Performance: UNWIND vs Individual MERGE

Neptune charges per I/O request. The difference between individual MERGE statements
and batched UNWIND queries is significant at scale:

| Method | Queries Sent | Neptune Cost | Latency |
|--------|-------------|-------------|--------|
| Individual MERGE | N (one per node) | N I/O requests | N × round-trip |
| Batch UNWIND | 1 (all nodes in one query) | 1 I/O request | 1 × round-trip |

For 1,000 nodes: individual = 1,000 queries. Batch = 1 query. Same result.

This notebook benchmarks both approaches and demonstrates the `batch_nodes_unwind`
and `batch_edges_unwind` functions that make Neptune writes cost-efficient.

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]`
- Neptune cluster endpoint
- For local-only benchmarking (Cypher generation without execution): base install only

## 1. Generate Test Data

In [ ]:
import time
from graphrag_toolkit.document_graph import Node, Edge
from graphrag_toolkit.document_graph.graph_build.cypher_builder import (
    node_to_cypher, batch_nodes_unwind,
    edge_to_cypher, batch_edges_unwind,
    batch_nodes_to_cypher, batch_edges_to_cypher,
)

# Generate N test nodes
N = 1000
TENANT = 'perf_test'

nodes = [
    Node(
        id=f'perf-node-{i:04d}',
        labels=['PerfTest'],
        properties={'index': i, 'batch': 'benchmark', 'value': f'data-{i}'}
    )
    for i in range(N)
]

edges = [
    Edge(
        id=f'perf-edge-{i:04d}',
        source_id=f'perf-node-{i:04d}',
        target_id=f'perf-node-{(i+1) % N:04d}',
        label='NEXT',
        properties={'order': i}
    )
    for i in range(N)
]

print(f'Generated {N} nodes and {N} edges for benchmarking')

## 2. Benchmark: Cypher Generation Speed

Compare the time to generate Cypher statements — individual vs batch.
This runs locally (no Neptune needed).

In [ ]:
# Individual MERGE — one query per node
start = time.perf_counter()
individual_queries = batch_nodes_to_cypher(nodes, tenant_id=TENANT)
individual_time = time.perf_counter() - start

# Batch UNWIND — one query for all nodes
start = time.perf_counter()
batch_query, batch_params = batch_nodes_unwind(nodes, tenant_id=TENANT)
batch_time = time.perf_counter() - start

print(f'Cypher generation for {N} nodes:')
print(f'  Individual: {individual_time*1000:.1f}ms → {len(individual_queries)} queries')
print(f'  Batch:      {batch_time*1000:.1f}ms → 1 query ({len(batch_params["batch"])} rows in UNWIND)')
print(f'\nQuery count reduction: {len(individual_queries)}× → 1×')
print(f'That\'s {len(individual_queries)}x fewer Neptune I/O requests')

## 3. Compare Generated Cypher

See exactly what each approach produces.

In [ ]:
print('=== Individual MERGE (first 3 of 1000) ===')
for q, p in individual_queries[:3]:
    print(f'  {q}')
    print(f'  params: {p}')
    print()

print(f'=== Batch UNWIND (single query for all {N}) ===')
print(f'  {batch_query}')
print(f'  batch[0]: {batch_params["batch"][0]}')
print(f'  batch[1]: {batch_params["batch"][1]}')
print(f'  ... ({len(batch_params["batch"])} total rows)')

## 4. Benchmark: Neptune Write (Optional)

If you have a Neptune endpoint, uncomment and run this cell to see the actual
wall-clock difference. Typical results:

- 1,000 individual MERGEs: ~30-60 seconds (network round-trips dominate)
- 1 UNWIND with 1,000 rows: ~1-3 seconds (single round-trip)

In [ ]:
import os

GRAPH_STORE = os.environ.get('GRAPH_STORE', '')

if GRAPH_STORE and '<your-' not in GRAPH_STORE:
    from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
    
    graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
    gs = graph_store.__enter__()
    
    # Batch write
    start = time.perf_counter()
    gs.execute_query(batch_query, batch_params)
    batch_write_time = time.perf_counter() - start
    print(f'Batch UNWIND ({N} nodes): {batch_write_time:.2f}s')
    
    # Individual write (first 100 only to save time)
    SAMPLE = 100
    start = time.perf_counter()
    for q, p in individual_queries[:SAMPLE]:
        gs.execute_query(q, p)
    individual_write_time = time.perf_counter() - start
    projected = individual_write_time * (N / SAMPLE)
    print(f'Individual MERGE ({SAMPLE} nodes): {individual_write_time:.2f}s (projected for {N}: {projected:.1f}s)')
    
    speedup = projected / batch_write_time
    print(f'\n⚡ Batch is ~{speedup:.0f}x faster')
    print(f'💰 Batch uses {N}x fewer I/O requests → proportional cost savings')
    
    # Cleanup test data
    gs.execute_query(f"MATCH (n:`PerfTest{TENANT}__`) DETACH DELETE n")
    print(f'\n🧹 Cleaned up {N} test nodes')
    graph_store.__exit__(None, None, None)
else:
    print('No Neptune endpoint configured — showing projected savings only.')
    print(f'\nProjected for {N} nodes:')
    print(f'  Individual: ~{N * 0.03:.0f}s ({N} round-trips × ~30ms each)')
    print(f'  Batch:      ~1-3s (1 round-trip)')
    print(f'  Speedup:    ~10-30x')
    print(f'  Cost:       {N}x fewer I/O requests')

## 5. Edge Batching

The same pattern works for edges — `batch_edges_unwind` writes all relationships
in a single UNWIND query using Neptune's native `~id` index for O(1) vertex lookups.

In [ ]:
# Individual edge queries
individual_edge_queries = batch_edges_to_cypher(edges[:10], tenant_id=TENANT)
print(f'Individual edge MERGE (sample):')
print(f'  {individual_edge_queries[0][0]}')

# Batch edge query
edge_query, edge_params = batch_edges_unwind(edges, tenant_id=TENANT)
print(f'\nBatch edge UNWIND ({len(edges)} edges in 1 query):')
print(f'  {edge_query}')
print(f'  batch rows: {len(edge_params["batch"])}')
print(f'\nNote: MATCH on ~id uses Neptune\'s native index — O(1) per vertex lookup')

## Summary

| Approach | Queries | Network Round-Trips | Neptune I/O Cost |
|----------|---------|--------------------|-----------------|
| `node_to_cypher` × N | N | N | N × per-request cost |
| `batch_nodes_unwind` | 1 | 1 | 1 × per-request cost |

**Always use batch functions for production writes:**
- `batch_nodes_unwind(nodes, tenant_id)` — all nodes in one UNWIND
- `batch_edges_unwind(edges, tenant_id)` — all edges in one UNWIND

Reserve individual MERGE for:
- Single-node updates (upserts)
- Debugging (readable query per node)
- Interactive notebook exploration